In [1]:
import json
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
from IPython.display import Markdown, display

In [2]:
# Команда запуску сервісу: py -m uvicorn app.main:app --reload
# Тестове сховище: In-memory (очищається при перезапуску процесу сервера)
# Для тестів використовуємо Mock замість реальної LLM

BASE_URL = "http://127.0.0.1:8000"
LIST_PATH = "/exercises"
MANUAL_PATH = "/exercises"
AI_PATH = "/exercises/from-text"
HTTP_TIMEOUT_SECONDS = 30

LIST_HTTP = 200
MANUAL_HTTP = 201
INVALID_MANUAL_HTTP = 422
CASE_RESULTS = []

In [3]:
def call_api(method, path, payload=None):
    data = None if payload is None else json.dumps(payload, ensure_ascii=False).encode("utf-8")
    request = Request(
        BASE_URL.rstrip("/") + path,
        data=data,
        headers={"Accept": "application/json", "Content-Type": "application/json"},
        method=method,
    )
    try:
        with urlopen(request, timeout=HTTP_TIMEOUT_SECONDS) as response:
            status, raw = response.status, response.read()
    except HTTPError as error:
        with error:
            status, raw = error.code, error.read()
    except (URLError, TimeoutError, OSError) as error:
        raise RuntimeError("HTTP-запит не завершився. Перевірте сервер, адресу й час очікування.") from error
    try:
        body = json.loads(raw.decode("utf-8"))
    except (UnicodeDecodeError, json.JSONDecodeError) as error:
        raise RuntimeError(f"HTTP {status}: відповідь не відповідає JSON-контракту.") from error
    print(f"{method} {path} → HTTP {status}")
    print(json.dumps(body, ensure_ascii=False, indent=2))
    return status, body

In [4]:
if "CASE_RESULTS" not in globals():
    CASE_RESULTS = []

def run_check(name, check):
    global CASE_RESULTS
    try:
        check()
    except Exception as error:
        row = {"scenario": name, "status": "failed", "reason": f"{type(error).__name__}: {error}"}
    else:
        row = {"scenario": name, "status": "passed", "reason": "Усі задані перевірки пройдено"}
    CASE_RESULTS.append(row)
    print(f"{row['scenario']}: {row['status']} — {row['reason']}\n")

def get_records():
    status, body = call_api("GET", LIST_PATH)
    assert status == LIST_HTTP, f"Очікували HTTP {LIST_HTTP}, отримали {status}"
    assert isinstance(body, list), "За контрактом список має бути JSON-масивом"
    assert all(isinstance(record, dict) and "id" in record for record in body)
    return body

def snapshot(records):
    ids = [record["id"] for record in records]
    assert len(ids) == len(set(ids)), "Ідентифікатори записів мають бути унікальними"
    return {record["id"]: record for record in records}

run_check("Список і початковий стан", get_records)

GET /exercises → HTTP 200
[]
Список і початковий стан: passed — Усі задані перевірки пройдено



In [5]:
MANUAL_INPUT = {"name": "Жим гантелей лежачи", "difficulty": "середній", "equipment": "гантелі"}
EXPECTED_FIELDS = {"name": "Жим гантелей лежачи", "difficulty": "середній", "equipment": "гантелі"}

def check_manual_create():
    before = snapshot(get_records())
    status, created = call_api("POST", MANUAL_PATH, MANUAL_INPUT)
    assert status == MANUAL_HTTP, f"Очікували HTTP {MANUAL_HTTP}, отримали {status}"
    assert isinstance(created, dict) and "id" in created, "Немає об’єкта запису з id"
    assert all(created.get(key) == value for key, value in EXPECTED_FIELDS.items()), "Хибні поля відповіді"
    assert created["id"] not in before, "Ідентифікатор нового запису вже існував"
    after = snapshot(get_records())
    assert len(after) == len(before) + 1, "Кількість записів не зросла рівно на один"
    assert created["id"] in after, "Створений запис відсутній у списку"
    assert after[created["id"]] == created, "Відповідь створення не збігається зі списком"
    assert all(after.get(key) == value for key, value in before.items()), "Попередні записи змінилися"

run_check("Ручне створення", check_manual_create)

GET /exercises → HTTP 200
[]
POST /exercises → HTTP 201
{
  "name": "Жим гантелей лежачи",
  "difficulty": "середній",
  "equipment": "гантелі",
  "id": 1
}
GET /exercises → HTTP 200
[
  {
    "name": "Жим гантелей лежачи",
    "difficulty": "середній",
    "equipment": "гантелі",
    "id": 1
  }
]
Ручне створення: passed — Усі задані перевірки пройдено



In [6]:
INVALID_MANUAL_INPUT = {"name": "   ", "difficulty": "початковий"}

def check_invalid_manual():
    before = snapshot(get_records())
    status, error_body = call_api("POST", MANUAL_PATH, INVALID_MANUAL_INPUT)
    assert status == INVALID_MANUAL_HTTP, f"Очікували HTTP {INVALID_MANUAL_HTTP}, отримали {status}"
    assert isinstance(error_body, dict) and error_body, "Немає опису відмови"
    assert snapshot(get_records()) == before, "Відхилений запит змінив записи"

run_check("Хибний ручний вхід", check_invalid_manual)

GET /exercises → HTTP 200
[
  {
    "name": "Жим гантелей лежачи",
    "difficulty": "середній",
    "equipment": "гантелі",
    "id": 1
  }
]
POST /exercises → HTTP 422
{
  "detail": "Назва вправи не може бути порожньою"
}
GET /exercises → HTTP 200
[
  {
    "name": "Жим гантелей лежачи",
    "difficulty": "середній",
    "equipment": "гантелі",
    "id": 1
  }
]
Хибний ручний вхід: passed — Усі задані перевірки пройдено



In [ ]:
# 1. Коректна AI-пропозиція
def check_ai_create():
    before = snapshot(get_records())
    status, created = call_api("POST", AI_PATH, {"text": "Звичайні віджимання від підлоги"})
    assert status == 201, f"Очікували HTTP 201, отримали {status}"
    assert created["name"] == "Віджимання від підлоги"
    after = snapshot(get_records())
    assert len(after) == len(before) + 1, "Запис не додано у каталог"

run_check("Коректна AI-пропозиція", check_ai_create)

# 2. Неповний або неоднозначний текст
def check_ai_incomplete():
    before = snapshot(get_records())
    status, error = call_api("POST", AI_PATH, {"text": "Якась вправа з гантелями"})
    assert status == 422, f"Очікували HTTP 422, отримали {status}"
    assert snapshot(get_records()) == before, "Сховище змінилося при неповній пропозиції"

run_check("Неповний або неоднозначний текст", check_ai_incomplete)

# 3. Структурно хибна пропозиція
def check_ai_structural():
    before = snapshot(get_records())
    status, error = call_api("POST", AI_PATH, {"text": "   "})
    assert status == 422, f"Очікували HTTP 422, отримали {status}"
    assert snapshot(get_records()) == before, "Сховище змінилося при порожньому тексті"

run_check("Структурно хибна пропозиція", check_ai_structural)

# 4. Змістовно хибна пропозиція (Галюцинація)
def check_ai_hallucinated():
    before = snapshot(get_records())
    status, error = call_api("POST", AI_PATH, {"text": "Випадання вперед з вагою"})
    assert status == 422, f"Очікували HTTP 422, отримали {status}"
    assert "не згадується" in error.get("detail", ""), "Немає повідомлення про відсутній інвентар"
    assert snapshot(get_records()) == before, "Сховище змінилося при галюцинації AI"

run_check("Змістовно хибна пропозиція", check_ai_hallucinated)

# 5. Збій моделі й незалежний ручний шлях
def check_ai_outage():
    call_api("POST", "/debug/toggle-outage")  # Симулюємо збій AI
    try:
        ai_status, _ = call_api("POST", AI_PATH, {"text": "Звичайні віджимання від підлоги"})
        assert ai_status == 503, f"Очікували HTTP 503 при збої AI, отримали {ai_status}"
        
        # Перевіряємо, що ручний ендпоінт далі працює
        man_status, created = call_api("POST", MANUAL_PATH, {"name": "Планка для пресу"})
        assert man_status == 201, "Ручний шлях не працює під час збою AI"
    finally:
        call_api("POST", "/debug/toggle-outage")  # Відновлюємо AI

run_check("Збій моделі й незалежний ручний шлях", check_ai_outage)

# 6. Жива інтеграція
CASE_RESULTS.append({
    "scenario": "Жива інтеграція через сервіс",
    "status": "not_run",
    "reason": "Використовується Mock-сервіс для стабільного тестування"
})

GET /exercises → HTTP 200
[
  {
    "name": "Жим гантелей лежачи",
    "difficulty": "середній",
    "equipment": "гантелі",
    "id": 1
  }
]
POST /exercises/from-text → HTTP 201
{
  "name": "Віджимання від підлоги",
  "difficulty": "початковий",
  "equipment": "без обладнання",
  "id": 2
}
GET /exercises → HTTP 200
[
  {
    "name": "Жим гантелей лежачи",
    "difficulty": "середній",
    "equipment": "гантелі",
    "id": 1
  },
  {
    "name": "Віджимання від підлоги",
    "difficulty": "початковий",
    "equipment": "без обладнання",
    "id": 2
  }
]
Коректна AI-пропозиція: passed — Усі задані перевірки пройдено

GET /exercises → HTTP 200
[
  {
    "name": "Жим гантелей лежачи",
    "difficulty": "середній",
    "equipment": "гантелі",
    "id": 1
  },
  {
    "name": "Віджимання від підлоги",
    "difficulty": "початковий",
    "equipment": "без обладнання",
    "id": 2
  }
]
POST /exercises/from-text → HTTP 422
{
  "detail": "Відсутня назва вправи (потрібне уточнення)"
}
GET /ex

In [8]:
from IPython.display import Markdown, display

if "CASE_RESULTS" not in globals():
    CASE_RESULTS = []

required = [
    "Список і початковий стан", "Ручне створення", "Хибний ручний вхід",
    "Коректна AI-пропозиція", "Неповний або неоднозначний текст",
    "Структурно хибна пропозиція", "Змістовно хибна пропозиція",
    "Збій моделі й незалежний ручний шлях", "Жива інтеграція через сервіс",
]

latest = {row["scenario"]: row for row in CASE_RESULTS}
rows = [latest.get(name, {"scenario": name, "status": "not_run", "reason": "Немає результату"}) for name in required]

table = "| Сценарій | Результат | Причина |\n|---|---|---|\n"
for row in rows:
    reason = row["reason"].replace("|", "/").replace("\n", " ")
    table += f"| {row['scenario']} | {row['status']} | {reason} |\n"

display(Markdown(table))

for status_val in ("passed", "failed", "not_run"):
    print(f"{status_val}: {sum(row['status'] == status_val for row in rows)}")

| Сценарій | Результат | Причина |
|---|---|---|
| Список і початковий стан | passed | Усі задані перевірки пройдено |
| Ручне створення | passed | Усі задані перевірки пройдено |
| Хибний ручний вхід | passed | Усі задані перевірки пройдено |
| Коректна AI-пропозиція | passed | Усі задані перевірки пройдено |
| Неповний або неоднозначний текст | passed | Усі задані перевірки пройдено |
| Структурно хибна пропозиція | passed | Усі задані перевірки пройдено |
| Змістовно хибна пропозиція | passed | Усі задані перевірки пройдено |
| Збій моделі й незалежний ручний шлях | passed | Усі задані перевірки пройдено |
| Жива інтеграція через сервіс | not_run | Використовується Mock-сервіс для стабільного та детермінованого тестування |


passed: 8
failed: 0
not_run: 1


Самоперевірка й висновок

Що довела відповідь сервера, а що довело порівняння зі специфікацією?
Відповідь сервера просто показала, що він піднімається, приймає запити і віддає JSON. А порівняння зі специфікацією підтвердило, що повертаються саме ті поля, які ми домовлялися (назва, складність, інвентар), і приходять правильні статус-коди типу 201 чи 422.

Чому незмінної кількості записів недостатньо для перевірки відмови?
Бо якщо сервіс випадково видалить старий запис і створить новий кривий, то загальна кількість залишиться такою ж, і ми не помітимо багу. Тому я порівнював усі записи по id до і після запиту, щоб переконатися, що старі дані точно не зачепило.

Який сценарій виявив проблему і що змінилося після виправлення?
Сценарій з галюцинаціями виявив проблему: спочатку сервер пропускав вигаданий інвентар (наприклад, гайку), і тест падав. Довелося дописати перевірку слів у services.py, щоб інвентар точно був у тексті користувача. Тепер сервер правильно відбиває такі запити з кодом 422.

Які результати отримані з підготовленими відповідями, а які з реальною моделлю?
Усі перевірки в ноутбуці зроблені через фейкову модель (заглушку), щоб відповіді завжди були однакові і тести не ламалися. Реальну модель я не підключав.

Чи є failed або not_run, і який доказ через це відсутній?
Так, є один not_run для сценарію Жива інтеграція. Я його пропустив, бо використовував заглушку, тому доказу роботи з реальним OpenAI API в цьому ноутбуці немає.